# 🕵️‍♂️ DeepFake Face Detection — Training & Evaluation Driver Notebook

This notebook acts as a thin driver that imports modular routines from the `src/` directory. It trains an **EfficientNet-B3** binary classifier to distinguish between **Authentic (Real)** and **AI-Generated (Fake)** facial images.

### 📁 Modular Architecture (`src/`):
- `src.face_detector.FaceDetector`: MTCNN-based face detection, alignment, and cropping pipeline.
- `src.dataset`: Advanced training augmentations (`JPEGCompression`, Gaussian blur, color jitter) to prevent model overfitting to dataset compression artifacts.
- `src.model`: EfficientNet-B3 network construction and weight serialization loader.
- `src.train`: Modular training, validation, and evaluation loops.
- `src.inference`: Standalone inference predictor with calibrated confidence thresholds.

---

In [ ]:
# ========================================================
# 1. Environment Setup & Module Imports
# ========================================================
import os
import random
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, ImageFolder

from src.face_detector import FaceDetector
from src.dataset import get_train_transforms, get_eval_transforms, PILListDataset
from src.model import build_efficientnet_b3, load_model_weights
from src.train import train_one_epoch, validate_one_epoch, evaluate_classifier
from src.inference import DeepFakePredictor, predict_single_image

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = torch.cuda.is_available()
CLASS_NAMES = ["fake", "real"]  # 0 = fake/AI-generated, 1 = real/authentic

print(f"Using PyTorch version: {torch.__version__}")
print(f"Execution Device:     {DEVICE}")
print(f"Class Order:          {CLASS_NAMES} (0=fake, 1=real)")

In [ ]:
# ========================================================
# 2. Face Detector Preview (MTCNN Cropping & Alignment)
# ========================================================
face_detector = FaceDetector(device=DEVICE)
print("Initialized FaceDetector module.")

# Demo face detection on a sample image if available
sample_path = "scratch/test_sample.jpg"
if os.path.exists(sample_path):
    sample_img = Image.open(sample_path)
    cropped_img, detected, bbox = face_detector.detect_and_crop(sample_img)
    
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(sample_img)
    axes[0].set_title("Original Input Image")
    axes[0].axis("off")
    
    axes[1].imshow(cropped_img)
    axes[1].set_title(f"MTCNN Cropped Face\n(Detected: {detected})")
    axes[1].axis("off")
    plt.show()

In [ ]:
# ========================================================
# 3. Dataset Preprocessing & Transform Pipeline
# ========================================================
IMG_SIZE = 224
BATCH_SIZE = 32

train_transform = get_train_transforms(img_size=IMG_SIZE)
eval_transform  = get_eval_transforms(img_size=IMG_SIZE)

print("Training Transforms:")
print(train_transform)

print("\nEvaluation Transforms:")
print(eval_transform)

In [ ]:
# ========================================================
# 4. Model Initialization & Weight Loading
# ========================================================
MODEL_PATH = "best_model_rebuilt"

model = build_efficientnet_b3(num_classes=2, dropout=0.4, pretrained=True)
model = model.to(DEVICE)

if os.path.exists(MODEL_PATH):
    try:
        model = load_model_weights(model, MODEL_PATH, DEVICE)
        print(f"Loaded trained model weights from: {MODEL_PATH}")
    except Exception as e:
        print(f"Could not load checkpoint: {e}. Starting with ImageNet weights.")
else:
    print("No pre-saved model found. Ready for training from scratch.")

In [ ]:
# ========================================================
# 5. Optional Fine-Tuning Routine
# ========================================================
# To run a complete training epoch, supply your dataset path below:
DATASET_ROOT = "/kaggle/input/140k-real-and-fake-faces"
EPOCHS = 5

if os.path.exists(os.path.join(DATASET_ROOT, "train")):
    train_dataset = ImageFolder(os.path.join(DATASET_ROOT, "train"), transform=train_transform)
    val_dataset   = ImageFolder(os.path.join(DATASET_ROOT, "valid"), transform=eval_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    
    print(f"Starting training on {len(train_dataset)} samples...")
    for epoch in range(1, EPOCHS + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE, use_amp=USE_AMP)
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, DEVICE, use_amp=USE_AMP)
        print(f"Epoch {epoch}/{EPOCHS} | Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc*100:.2f}% | Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%")
else:
    print("Dataset root not found locally. Skipping inline training step. Use model checkpoint for evaluation.")

In [ ]:
# ========================================================
# 6. In-Distribution vs Out-of-Distribution (OOD) Evaluation
# ========================================================
# Evaluate on Kaggle In-Distribution test split if available
test_split_path = os.path.join(DATASET_ROOT, "test")
if os.path.exists(test_split_path):
    test_dataset = ImageFolder(test_split_path, transform=eval_transform)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    id_metrics, id_labels, id_preds, id_probs = evaluate_classifier(
        model, test_loader, DEVICE, class_names=CLASS_NAMES, dataset_name="In-Distribution Kaggle Test Set"
    )
else:
    print("Kaggle test set not found locally. Demonstrating evaluation API.")

In [ ]:
# ========================================================
# 7. Custom Image Prediction with Standalone Inference
# ========================================================
predictor = DeepFakePredictor(model_path=MODEL_PATH, device=DEVICE, threshold=0.5)

# Analyze sample image
if os.path.exists(sample_path):
    result = predictor.predict(sample_path)
    print(f"Sample Result: {result['predicted_class']} ({result['confidence_percentage']:.2f}% confidence)")
    predict_single_image(sample_path, model_path=MODEL_PATH, device=DEVICE, show=True)